In [ ]:

import pandas as pd
import numpy as np
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

uploaded = files.upload()
df = pd.read_csv("appointments_clean.csv")

print("Dataset shape:", df.shape)

X = df[[
    "age", "distance_km", "fee_inr", "lead_days",
    "prior_no_shows", "reminder_sent",
    "booking_channel", "visit_type"
]].copy()

X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})

X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

y = df["no_show"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline_accuracy = (y_test == 0).mean()

print("Number of features:", X.shape[1])
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Test no-shows:", y_test.sum())
print("Baseline accuracy:", round(baseline_accuracy, 4))


Saving appointments_clean.csv to appointments_clean.csv
Dataset shape: (990, 16)
Number of features: 11
Training rows: 792
Test rows: 198
Test no-shows: 49
Baseline accuracy: 0.7525


In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score,
    f1_score, roc_auc_score
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:, 1]

print("Accuracy:", accuracy_score(y_test, lr_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, lr_pred))
print("Precision:", precision_score(y_test, lr_pred, zero_division=0))
print("Recall:", recall_score(y_test, lr_pred, zero_division=0))
print("F1:", f1_score(y_test, lr_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, lr_prob))

coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lr.coef_[0]
})
print("\nCoefficients:")
display(coef_table)

idx = X.columns.get_loc("reminder_sent")
coef = lr.coef_[0][idx]
scale = scaler.scale_[idx]

print("Odds ratio per 1 SD:", np.exp(coef))
print("Odds ratio for No to Yes:", np.exp(coef / scale))
print("Reminder association: the No-to-Yes odds ratio holds other features constant; it does not prove causation.")


Accuracy: 0.7878787878787878
Confusion Matrix:
[[139  10]
 [ 32  17]]
Precision: 0.6296296296296297
Recall: 0.3469387755102041
F1: 0.4473684210526316
ROC-AUC: 0.788933022873579

Coefficients:


,feature,coefficient
0,age,-0.253880
1,distance_km,0.253448
2,fee_inr,0.452761
3,lead_days,0.614471
4,prior_no_shows,0.649612
5,reminder_sent,-0.580669
6,booking_channel_App,-0.046804
7,booking_channel_Phone,0.215881
8,booking_channel_Walk-In,-0.186364
9,visit_type_Consultation,0.015658


Odds ratio per 1 SD: 0.559524141750829
Odds ratio for No to Yes: 0.3093980903124433
Reminder association: the No-to-Yes odds ratio holds other features constant; it does not prove causation.


In [ ]:

from sklearn.tree import DecisionTreeClassifier

tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

tree_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_3.fit(X_train, y_train)

print("Full-depth tree")
print("Training accuracy:", tree_full.score(X_train, y_train))
print("Test accuracy:", tree_full.score(X_test, y_test))
print("Tree depth:", tree_full.get_depth())

print("\nDepth-3 tree")
print("Training accuracy:", tree_3.score(X_train, y_train))
print("Test accuracy:", tree_3.score(X_test, y_test))

print("\nA large train-test gap suggests overfitting: the model memorises training data but performs worse on unseen bookings.")


Full-depth tree
Training accuracy: 1.0
Test accuracy: 0.6717171717171717
Tree depth: 15

Depth-3 tree
Training accuracy: 0.821969696969697
Test accuracy: 0.7777777777777778

A large train-test gap suggests overfitting: the model memorises training data but performs worse on unseen bookings.


In [ ]:

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("Random Forest Recall:", recall_score(y_test, rf_pred))
print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_prob))

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 3 important features:")
display(importance.head(3))

tree_pred = tree_3.predict(X_test)
tree_prob = tree_3.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Depth-3 Tree", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, tree_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "Recall": [
        recall_score(y_test, lr_pred),
        recall_score(y_test, tree_pred),
        recall_score(y_test, rf_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, lr_prob),
        roc_auc_score(y_test, tree_prob),
        roc_auc_score(y_test, rf_prob)
    ]
})

print("\nModel comparison (reporting only):")
display(comparison.round(4))

print("The highest test ROC-AUC is shown for reporting only.")
print("The final model must be chosen using training cross-validation in Task 3.5.")


Random Forest Accuracy: 0.7474747474747475
Random Forest Recall: 0.30612244897959184
Random Forest ROC-AUC: 0.7057252431173813

Top 3 important features:


,feature,importance
3,lead_days,0.219905
0,age,0.196788
1,distance_km,0.194739



Model comparison (reporting only):


,Model,Accuracy,Recall,ROC-AUC
0,Logistic Regression,0.7879,0.3469,0.7889
1,Depth-3 Tree,0.7778,0.2245,0.6997
2,Random Forest,0.7475,0.3061,0.7057


The highest test ROC-AUC is shown for reporting only.
The final model must be chosen using training cross-validation in Task 3.5.


In [ ]:

from sklearn.model_selection import (
    GridSearchCV, cross_val_score, cross_val_predict
)
from sklearn.pipeline import Pipeline

# (a) Tune Random Forest using training data only
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    cv=5,
    scoring="roc_auc"
)

grid.fit(X_train, y_train)

print("Best RF parameters:", grid.best_params_)
print("Best RF CV ROC-AUC:", grid.best_score_)

# (b) Compare models using training cross-validation
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

models = {
    "Logistic Regression": lr_pipe,
    "Depth-3 Tree": DecisionTreeClassifier(
        max_depth=3, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42
    )
}

cv_results = {}

for name, model in models.items():
    score = cross_val_score(
        model, X_train, y_train,
        cv=5, scoring="roc_auc"
    ).mean()
    cv_results[name] = score

cv_results["Tuned Random Forest"] = grid.best_score_

print("\nTraining CV ROC-AUC:")
display(pd.DataFrame({
    "Model": list(cv_results.keys()),
    "CV ROC-AUC": list(cv_results.values())
}).round(4))

winner_name = max(cv_results, key=cv_results.get)
print("Chosen model:", winner_name)

if winner_name == "Tuned Random Forest":
    winner = grid.best_estimator_
else:
    winner = models[winner_name]

# (c) Choose threshold using out-of-fold training probabilities
oof_probs = cross_val_predict(
    winner, X_train, y_train,
    cv=5, method="predict_proba"
)[:, 1]

threshold_results = []

for threshold in [0.5, 0.35]:
    pred = (oof_probs >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_train, pred, zero_division=0),
        "Recall": recall_score(y_train, pred, zero_division=0),
        "F1": f1_score(y_train, pred, zero_division=0),
        "Flagged": int(pred.sum())
    })

threshold_table = pd.DataFrame(threshold_results)

print("\nTraining threshold comparison:")
display(threshold_table.round(4))

best_threshold = float(
    threshold_table.loc[
        threshold_table["F1"].idxmax(), "Threshold"
    ]
)

print("Chosen threshold:", best_threshold)
print("Higher F1 balances precision and recall.")
print("Recall matters because a missed warning costs an empty appointment slot, while an extra call costs staff time.")

# (d) Fit final winner and evaluate on test set once
winner.fit(X_train, y_train)

test_probs = winner.predict_proba(X_test)[:, 1]
test_flags = (test_probs >= best_threshold).astype(int)

print("\nFINAL TEST RESULTS")
print("Model:", winner_name)
print("Threshold:", best_threshold)
print("Precision:", precision_score(y_test, test_flags, zero_division=0))
print("Recall:", recall_score(y_test, test_flags, zero_division=0))
print("Number flagged:", int(test_flags.sum()))
print("ROC-AUC:", roc_auc_score(y_test, test_probs))

# (e) Save test risk scores
test_risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": test_probs,
    "flag": test_flags
})

test_risk_scores.to_csv("test_risk_scores.csv", index=False)

print("\nSaved rows:", len(test_risk_scores))
display(test_risk_scores.head())

print("\nIn practice, the clinic should choose its threshold based on how many follow-up calls staff can make each evening compared with the cost of an empty appointment slot.")

files.download("test_risk_scores.csv")


Best RF parameters: {'max_depth': 3, 'n_estimators': 200}
Best RF CV ROC-AUC: 0.7673242251739405

Training CV ROC-AUC:


,Model,CV ROC-AUC
0,Logistic Regression,0.7737
1,Depth-3 Tree,0.6784
2,Random Forest,0.7466
3,Tuned Random Forest,0.7673


Chosen model: Logistic Regression

Training threshold comparison:


,Threshold,Precision,Recall,F1,Flagged
0,0.50,0.5753,0.2485,0.3471,73
1,0.35,0.5130,0.4675,0.4892,154


Chosen threshold: 0.35
Higher F1 balances precision and recall.
Recall matters because a missed warning costs an empty appointment slot, while an extra call costs staff time.

FINAL TEST RESULTS
Model: Logistic Regression
Threshold: 0.35
Precision: 0.5714285714285714
Recall: 0.5714285714285714
Number flagged: 49
ROC-AUC: 0.788933022873579

Saved rows: 198


,appointment_id,no_show,risk_prob,flag
0,A0217,0,0.021565,0
1,A0336,0,0.150917,0
2,A0508,0,0.518692,1
3,A0314,0,0.191300,0
4,A0089,0,0.167718,0



In practice, the clinic should choose its threshold based on how many follow-up calls staff can make each evening compared with the cost of an empty appointment slot.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>